In [1]:
import pandas as pd

In [2]:
DATA_PATH = "final_crm_dashboard_dataset_with_only_3_nlp_fields.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())

Shape: (15000, 18)
  interaction_id crm_note_id reference_sentiment        reference_topics  \
0      INT000001   CRM000001               Mixed            Tolerability   
1      INT000002   CRM000002               Mixed               Adherence   
2      INT000003   CRM000003               Mixed  Access / Reimbursement   
3      INT000004   CRM000004               Mixed    Cost / Affordability   
4      INT000005   CRM000005            Negative  Access / Reimbursement   

                   reference_objections interaction_date   rep_id    hcp_id  \
0           Side Effects / Tolerability       01-01-2026  REP0002  HCP00001   
1                     Adherence Concern       28-02-2026  REP0002  HCP00001   
2  Efficacy Concern|Prior Authorization       04-03-2026  REP0002  HCP00001   
3               High Out-of-Pocket Cost       12-03-2026  REP0002  HCP00001   
4                   Prior Authorization       18-03-2026  REP0002  HCP00001   

        city region       territory_id hcp_specia

In [3]:
print(df.columns.tolist())

['interaction_id', 'crm_note_id', 'reference_sentiment', 'reference_topics', 'reference_objections', 'interaction_date', 'rep_id', 'hcp_id', 'city', 'region', 'territory_id', 'hcp_specialization', 'drug_id', 'drug_name', 'crm_note', 'topic', 'sentiment', 'objection']


In [ ]:
#Prepare the date
df["interaction_date"] = pd.to_datetime(
    df["interaction_date"],
    dayfirst=True,
    errors="coerce"
)

df["month"] = (
    df["interaction_date"]
    .dt.to_period("M")
    .astype(str)
)

print(df[["interaction_date", "month"]].head())

In [5]:
#Clean sentiment
df["sentiment"] = (
    df["sentiment"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(df["sentiment"].value_counts())

sentiment
mixed       7491
negative    3864
positive    3021
neutral      624
Name: count, dtype: int64


In [6]:
#Prepare topics
topic_df = df.copy()

topic_df["topic"] = (
    topic_df["topic"]
    .fillna("")
    .astype(str)
    .str.split("|")
)

topic_df = topic_df.explode("topic")

topic_df["topic"] = (
    topic_df["topic"]
    .str.strip()
)

topic_df = topic_df[
    topic_df["topic"] != ""
]

print(topic_df[["topic", "sentiment"]].head(10))

                    topic sentiment
0            Tolerability     mixed
0                  Safety     mixed
0                Efficacy     mixed
1            Tolerability     mixed
1               Adherence     mixed
2               Adherence     mixed
2                Efficacy     mixed
2  Access / Reimbursement     mixed
3                Efficacy     mixed
3                    Cost     mixed


In [7]:
#Prepare objections
objection_df = df.copy()

objection_df["objection"] = (
    objection_df["objection"]
    .fillna("")
    .astype(str)
    .str.split("|")
)

objection_df = objection_df.explode(
    "objection"
)

objection_df["objection"] = (
    objection_df["objection"]
    .str.strip()
)

objection_df = objection_df[
    objection_df["objection"] != ""
]

print(
    objection_df[
        ["drug_name", "objection", "sentiment"]
    ].head(10)
)

   drug_name                    objection sentiment
0  Arthrelis  Side Effects / Tolerability     mixed
1  Arthrelis            Adherence Concern     mixed
2  Arthrelis             Efficacy Concern     mixed
2  Arthrelis          Prior Authorization     mixed
3  Arthrelis      High Out-of-Pocket Cost     mixed
4  Rheumorel          Prior Authorization  negative
5  Arthrelis        Clinical Evidence Gap     mixed
6  Arthrelis        Clinical Evidence Gap     mixed
7  Arthrelis               Safety Concern  negative
8  Rheumorel                 No Objection  positive


In [8]:
#Most common objections
common_objections = (
    objection_df
    .groupby("objection")
    .size()
    .reset_index(name="frequency")
    .sort_values(
        "frequency",
        ascending=False
    )
)

print(common_objections.head(10))

                      objection  frequency
8                  No Objection       3645
10          Prior Authorization       1753
7       High Out-of-Pocket Cost       1492
12  Side Effects / Tolerability       1451
4              Efficacy Concern       1237
0             Adherence Concern       1206
1         Clinical Evidence Gap       1196
2         Competitor Preference       1148
9             Poor Availability       1137
11               Safety Concern       1068


In [9]:
#Emerging topics
monthly_topics = (
    topic_df
    .groupby(["month", "topic"])
    .size()
    .reset_index(name="frequency")
)

print(monthly_topics.head(10))

     month                    topic  frequency
0  2026-01                   Access        121
1  2026-01       Access / Formulary        118
2  2026-01   Access / Reimbursement        218
3  2026-01                Adherence        134
4  2026-01             Availability        135
5  2026-01        Clinical Evidence        361
6  2026-01               Competitor        140
7  2026-01                     Cost        194
8  2026-01                   Dosing        177
9  2026-01  Dosing / Administration        114


In [10]:
months = sorted(
    topic_df["month"].dropna().unique()
)

print(months)

['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08']


In [11]:
if len(months) >= 2:

    current_month = months[-1]
    previous_month = months[-2]

    print("Current month:", current_month)
    print("Previous month:", previous_month)

Current month: 2026-08
Previous month: 2026-07


In [12]:
#Calculate topic growth
current = (
    topic_df[
        topic_df["month"] == current_month
    ]
    .groupby("topic")
    .size()
    .rename("current_frequency")
)

previous = (
    topic_df[
        topic_df["month"] == previous_month
    ]
    .groupby("topic")
    .size()
    .rename("previous_frequency")
)

trend = pd.concat(
    [current, previous],
    axis=1
).fillna(0)

print(trend)

                         current_frequency  previous_frequency
topic                                                         
Access                                  83                 167
Access / Formulary                      97                 194
Access / Reimbursement                 171                 330
Adherence                              134                 228
Availability                           116                 227
Clinical Evidence                      241                 517
Competitor                             137                 232
Cost                                   154                 294
Dosing                                 110                 208
Dosing / Administration                 87                 200
Ease of Use                            118                 228
Efficacy                               297                 625
General                                 28                  57
Patient Experience                     128             

In [13]:
def calculate_growth(row):

    if row["previous_frequency"] == 0:

        if row["current_frequency"] > 0:
            return 100

        return 0

    return (
        (
            row["current_frequency"]
            - row["previous_frequency"]
        )
        / row["previous_frequency"]
    ) * 100

In [14]:
trend["growth_percentage"] = trend.apply(
    calculate_growth,
    axis=1
)

trend = trend.reset_index()

print(trend.sort_values(
    "growth_percentage",
    ascending=False
).head(10))

                     topic  current_frequency  previous_frequency  \
6               Competitor                137                 232   
3                Adherence                134                 228   
13      Patient Experience                128                 233   
8                   Dosing                110                 208   
7                     Cost                154                 294   
2   Access / Reimbursement                171                 330   
10             Ease of Use                118                 228   
14       Patient Selection                 89                 172   
4             Availability                116                 227   
1       Access / Formulary                 97                 194   

    growth_percentage  
6          -40.948276  
3          -41.228070  
13         -45.064378  
8          -47.115385  
7          -47.619048  
2          -48.181818  
10         -48.245614  
14         -48.255814  
4          -48.898678  


In [15]:
#Create the emerging issue rule
emerging_issues = trend[
    (trend["growth_percentage"] >= 30) &
    (trend["current_frequency"] >= 5)
].sort_values(
    "growth_percentage",
    ascending=False
)

print(emerging_issues)

Empty DataFrame
Columns: [topic, current_frequency, previous_frequency, growth_percentage]
Index: []


This gives you:

Emerging issues

In [16]:
#Negative sentiment drivers
negative_drivers = (
    topic_df
    .groupby("topic")
    .agg(
        frequency=("topic", "size"),

        negative_count=(
            "sentiment",
            lambda x:
            (x == "negative").sum()
        )
    )
    .reset_index()
)

In [17]:
negative_drivers["negative_percentage"] = (
    negative_drivers["negative_count"]
    / negative_drivers["frequency"]
    * 100
)

In [18]:
negative_drivers = negative_drivers[
    (negative_drivers["frequency"] >= 20) &
    (negative_drivers["negative_percentage"] >= 60)
].sort_values(
    "negative_percentage",
    ascending=False
)

print(negative_drivers)

Empty DataFrame
Columns: [topic, frequency, negative_count, negative_percentage]
Index: []


In [19]:
#Brand-specific challenges
brand_objections = (
    objection_df
    .groupby(
        ["drug_name", "objection"]
    )
    .size()
    .reset_index(
        name="frequency"
    )
)

print(brand_objections.head(10))

    drug_name                objection  frequency
0  Acetovarin        Adherence Concern         67
1  Acetovarin    Clinical Evidence Gap         96
2  Acetovarin    Competitor Preference         72
3  Acetovarin        Dosing Complexity         57
4  Acetovarin         Efficacy Concern         76
5  Acetovarin     Eligibility Criteria         66
6  Acetovarin    Formulary Restriction         60
7  Acetovarin  High Out-of-Pocket Cost        127
8  Acetovarin             No Objection        251
9  Acetovarin        Poor Availability         75


In [20]:
brand_objections["brand_total"] = (
    brand_objections
    .groupby("drug_name")["frequency"]
    .transform("sum")
)

brand_objections["share_percentage"] = (
    brand_objections["frequency"]
    / brand_objections["brand_total"]
    * 100
)

In [21]:
brand_challenges = brand_objections[
    brand_objections["share_percentage"] >= 30
].sort_values(
    "share_percentage",
    ascending=False
)

print(brand_challenges)

Empty DataFrame
Columns: [drug_name, objection, frequency, brand_total, share_percentage]
Index: []


In [22]:
#Positive opportunities
positive_opportunities = (
    topic_df
    .groupby("topic")
    .agg(
        frequency=("topic", "size"),

        positive_count=(
            "sentiment",
            lambda x:
            (x == "positive").sum()
        )
    )
    .reset_index()
)

In [23]:
positive_opportunities["positive_percentage"] = (
    positive_opportunities["positive_count"]
    / positive_opportunities["frequency"]
    * 100
)

In [24]:
positive_opportunities = positive_opportunities[
    (positive_opportunities["frequency"] >= 20) &
    (positive_opportunities["positive_percentage"] >= 60)
].sort_values(
    "positive_percentage",
    ascending=False
)

print(positive_opportunities)

Empty DataFrame
Columns: [topic, frequency, positive_count, positive_percentage]
Index: []


In [25]:
#High-priority signals 🔴
priority = (
    objection_df
    .groupby(
        ["drug_name", "objection"]
    )
    .agg(
        frequency=("objection", "size"),

        negative_percentage=(
            "sentiment",
            lambda x:
            (x == "negative").mean() * 100
        )
    )
    .reset_index()
)

In [74]:
high_priority = priority[
    (priority["frequency"] >= 30) &
    (priority["negative_percentage"] >= 50)
].copy()

In [75]:
print("Number of high priority signals:",
      len(high_priority))

Number of high priority signals: 3


In [76]:
print(
    high_priority[
        [
            "drug_name",
            "objection",
            "frequency",
            "negative_percentage"
        ]
    ]
    .sort_values(
        "frequency",
        ascending=False
    )
    .head(20)
)

     drug_name            objection  frequency  negative_percentage
243  Infectrel    Poor Availability         48                 50.0
270    Liponex  Prior Authorization         46                 50.0
48    Alleriva    Poor Availability         32                 50.0


In [77]:
#Add business actions
BUSINESS_ACTIONS = {

    "Prior Authorization":
        "Review authorization barriers and assess whether additional HCP access support is warranted.",

    "Formulary Restriction":
        "Assess formulary and coverage barriers affecting patient access.",

    "High Out-of-Pocket Cost":
        "Assess affordability barriers and available patient-support options.",

    "Poor Availability":
        "Investigate supply and dispensing barriers affecting treatment availability.",

    "Safety Concern":
        "Review safety communication and address key HCP questions.",

    "Efficacy Concern":
        "Assess whether additional clinical evidence or HCP education could address efficacy concerns.",

    "Side Effects / Tolerability":
        "Assess whether additional tolerability education or HCP support is warranted.",

    "Adherence Concern":
        "Consider adherence-support resources and strategies to improve treatment persistence.",

    "Dosing Complexity":
        "Assess whether clearer dosing guidance could reduce administration complexity.",

    "Competitor Preference":
        "Review differentiation and messaging relative to competing therapies."
}

In [78]:
#Generate the actual cards
dashboard_cards = []

for _, row in high_priority.iterrows():

    objection = row["objection"]

    action = BUSINESS_ACTIONS.get(
        objection,
        "Investigate the underlying HCP barrier and assess targeted support."
    )

    card = {
        "card_type": "HIGH PRIORITY",
        "brand": row["drug_name"],
        "objection": objection,
        "frequency": int(row["frequency"]),
        "negative_sentiment": round(
            row["negative_percentage"],
            1
        ),
        "business_action": action
    }

    dashboard_cards.append(card)

In [81]:
#Display the cards
for card in dashboard_cards:

    print("\n" + "=" * 60)

    print("🔴 HIGH PRIORITY")

    print("Brand:", card["brand"])
    print("Objection:", card["objection"])

    print(
        "HCP interactions:",
        card["frequency"]
    )

    print(
        "Negative sentiment:",
        f"{card['negative_sentiment']}%"
    )

    print(
        "\nBusiness implication:"
    )

    print(
        f"{card['objection']} is a notable "
        f"barrier appearing frequently in HCP "
        f"interactions for {card['brand']}."
    )

    print(
        "\nPotential business action:"
    )

    print(
        card["business_action"]
    )


🔴 HIGH PRIORITY
Brand: Alleriva
Objection: Poor Availability
HCP interactions: 32
Negative sentiment: 50.0%

Business implication:
Poor Availability is a notable barrier appearing frequently in HCP interactions for Alleriva.

Potential business action:
Investigate supply and dispensing barriers affecting treatment availability.

🔴 HIGH PRIORITY
Brand: Infectrel
Objection: Poor Availability
HCP interactions: 48
Negative sentiment: 50.0%

Business implication:
Poor Availability is a notable barrier appearing frequently in HCP interactions for Infectrel.

Potential business action:
Investigate supply and dispensing barriers affecting treatment availability.

🔴 HIGH PRIORITY
Brand: Liponex
Objection: Prior Authorization
HCP interactions: 46
Negative sentiment: 50.0%

Business implication:
Prior Authorization is a notable barrier appearing frequently in HCP interactions for Liponex.

Potential business action:
Review authorization barriers and assess whether additional HCP access support i

This is important because later your backend can read this file.

In [82]:
#Save the insights
import json


In [83]:
#Save the cards
with open(
    "high_priority_insights.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dashboard_cards,
        f,
        indent=2,
        ensure_ascii=False
    )

print("high_priority_insights.json created successfully!")

high_priority_insights.json created successfully!


In [84]:
#Check the JSON
with open(
    "high_priority_insights.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(json.dumps(
    data,
    indent=2,
    ensure_ascii=False
))

[
  {
    "card_type": "HIGH PRIORITY",
    "brand": "Alleriva",
    "objection": "Poor Availability",
    "frequency": 32,
    "negative_sentiment": 50.0,
    "business_action": "Investigate supply and dispensing barriers affecting treatment availability."
  },
  {
    "card_type": "HIGH PRIORITY",
    "brand": "Infectrel",
    "objection": "Poor Availability",
    "frequency": 48,
    "negative_sentiment": 50.0,
    "business_action": "Investigate supply and dispensing barriers affecting treatment availability."
  },
  {
    "card_type": "HIGH PRIORITY",
    "brand": "Liponex",
    "objection": "Prior Authorization",
    "frequency": 46,
    "negative_sentiment": 50.0,
    "business_action": "Review authorization barriers and assess whether additional HCP access support is warranted."
  }
]
